# Statistics Advance-6 Assignment
### Data Science Masters - PW Skills

## Q1. Assumptions required to use ANOVA + examples of violations

**Assumptions of ANOVA:**

1. **Normality**: Each group's data should be approximately normally distributed.
   - Violation Example: Highly skewed income data across groups — ANOVA results become unreliable.

2. **Homogeneity of Variance (Homoscedasticity)**: All groups should have approximately equal variances.
   - Violation Example: Group A has std=2, Group B has std=20 — unequal variances inflate F-statistic.

3. **Independence**: Observations must be independent of each other.
   - Violation Example: Using the same students in multiple groups, or repeated measurements treated as independent.

4. **Random Sampling**: Data must be randomly sampled from the populations.
   - Violation Example: Convenience sampling (e.g., only surveying friends) introduces selection bias.

5. **Interval/Ratio Scale**: The dependent variable must be measured on a continuous scale.
   - Violation Example: Using ordinal data (like Likert scale 1-5) directly in ANOVA without caution.

**Impact of Violations:**
- Non-normality with small n → inflated Type I error rate.
- Unequal variances → invalid F-statistic (use Welch's ANOVA instead).
- Non-independence → severe inflation of false positive rate.

## Q2. Three types of ANOVA and when to use each

**1. One-Way ANOVA:**
- Used when comparing means across **3 or more groups** based on **one independent variable (factor)**.
- Example: Comparing average test scores across three teaching methods (A, B, C).

**2. Two-Way ANOVA:**
- Used when there are **two independent variables (factors)** and we want to study:
  - Main effect of each factor
  - Interaction effect between the two factors
- Example: Studying effect of diet type (A, B, C) AND exercise level (low, high) on weight loss simultaneously.

**3. Repeated Measures ANOVA:**
- Used when the **same subjects are measured multiple times** (within-subjects design).
- Accounts for correlation between repeated measurements from same individual.
- Example: Measuring blood pressure of the same patients before, during, and after treatment.

| Type | Factors | Subjects | Use Case |
|---|---|---|---|
| One-Way | 1 | Different | Compare 3+ groups on 1 factor |
| Two-Way | 2 | Different | Study 2 factors + interaction |
| Repeated Measures | 1+ | Same | Same subject, multiple timepoints |

## Q3. Partitioning of Variance in ANOVA

**Partitioning of Variance:**
ANOVA splits the total variation in data into two parts:

**SST = SSB + SSW**

- **SST (Total Sum of Squares)**: Total variation in the data from the grand mean.
- **SSB (Between-Group SS)**: Variation due to differences between group means — explained by the factor.
- **SSW (Within-Group SS / Residual SS)**: Variation within each group — unexplained, due to random error.

**F-statistic = MSB / MSW** where MS = SS / df

**Why is this important?**
1. Helps understand **how much of the total variation is explained** by the grouping factor.
2. Large SSB relative to SSW → groups are truly different → high F-statistic → reject H₀.
3. Foundation of effect size measures like **η² (Eta-squared)** = SSB/SST.
4. Without partitioning, we can't separate signal (group differences) from noise (random error).

## Q4. Calculate SST, SSE (Between), SSR (Within) in One-Way ANOVA using Python

In [1]:
import numpy as np
from scipy import stats

np.random.seed(42)

# Three groups
group_A = np.array([5, 6, 7, 8, 6])
group_B = np.array([9, 10, 11, 9, 10])
group_C = np.array([3, 4, 5, 4, 3])

all_data   = np.concatenate([group_A, group_B, group_C])
grand_mean = np.mean(all_data)
groups     = [group_A, group_B, group_C]

# SST — Total Sum of Squares
SST = np.sum((all_data - grand_mean) ** 2)

# SSB — Between Group (Explained) Sum of Squares
SSB = sum(len(g) * (np.mean(g) - grand_mean)**2 for g in groups)

# SSW — Within Group (Residual) Sum of Squares
SSW = sum(np.sum((g - np.mean(g))**2) for g in groups)

# Degrees of freedom
k   = len(groups)
N   = len(all_data)
df_between = k - 1
df_within  = N - k

MSB = SSB / df_between
MSW = SSW / df_within
F   = MSB / MSW

print("===== One-Way ANOVA — Sum of Squares =====")
print(f"Grand Mean : {grand_mean:.4f}")
print(f"\nSST (Total)          : {SST:.4f}")
print(f"SSB (Between/Explained): {SSB:.4f}")
print(f"SSW (Within/Residual)  : {SSW:.4f}")
print(f"\nVerification: SSB + SSW = {SSB + SSW:.4f} (should equal SST = {SST:.4f})")
print(f"\nMSB = {MSB:.4f}, MSW = {MSW:.4f}")
print(f"F-statistic = {F:.4f}")

# Verify with scipy
f_scipy, p_scipy = stats.f_oneway(group_A, group_B, group_C)
print(f"\nscipy F-statistic = {f_scipy:.4f}, p-value = {p_scipy:.6f}")

===== One-Way ANOVA — Sum of Squares =====
Grand Mean : 6.6667

SST (Total)          : 101.3333
SSB (Between/Explained): 90.5333
SSW (Within/Residual)  : 10.8000

Verification: SSB + SSW = 101.3333 (should equal SST = 101.3333)

MSB = 45.2667, MSW = 0.9000
F-statistic = 50.2963

scipy F-statistic = 50.2963, p-value = 0.000001


## Q5. Two-Way ANOVA — Main effects and Interaction effects using Python

In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.formula.api import ols

np.random.seed(42)

# Simulated data: Diet (A/B/C) x Exercise (Low/High) on weight loss
n = 10
data = pd.DataFrame({
    'WeightLoss': np.concatenate([
        np.random.normal(5, 1, n),   # Diet A, Low
        np.random.normal(7, 1, n),   # Diet A, High
        np.random.normal(6, 1, n),   # Diet B, Low
        np.random.normal(9, 1, n),   # Diet B, High
        np.random.normal(4, 1, n),   # Diet C, Low
        np.random.normal(6, 1, n),   # Diet C, High
    ]),
    'Diet': ['A']*n*2 + ['B']*n*2 + ['C']*n*2,
    'Exercise': (['Low']*n + ['High']*n) * 3
})

# Two-way ANOVA with interaction
model  = ols('WeightLoss ~ C(Diet) + C(Exercise) + C(Diet):C(Exercise)', data=data).fit()
anova_table = sm.stats.anova_lm(model, typ=2)

print("===== Two-Way ANOVA Table =====")
print(anova_table)
print("\nInterpretation:")
print("  C(Diet)              → Main effect of Diet")
print("  C(Exercise)          → Main effect of Exercise")
print("  C(Diet):C(Exercise)  → Interaction effect between Diet & Exercise")

===== Two-Way ANOVA Table =====
                        sum_sq    df          F        PR(>F)
C(Diet)              52.116417   2.0  35.728304  1.304085e-10
C(Exercise)          62.531782   1.0  85.737072  9.688628e-13
C(Diet):C(Exercise)  12.825002   2.0   8.792154  4.949372e-04
Residual             39.384553  54.0        NaN           NaN

Interpretation:
  C(Diet)              → Main effect of Diet
  C(Exercise)          → Main effect of Exercise
  C(Diet):C(Exercise)  → Interaction effect between Diet & Exercise


## Q6. Interpret One-Way ANOVA results: F=5.23, p=0.02

**Given:** F-statistic = 5.23, p-value = 0.02, α = 0.05

**Conclusion:**

Since p-value (0.02) < α (0.05), we **REJECT the null hypothesis**.

**Interpretation:**
- H₀ was: All group means are equal (μ₁ = μ₂ = μ₃ = ...)
- There is **statistically significant evidence** that at least one group mean is different from the others.
- The F-statistic of 5.23 indicates the between-group variance is 5.23 times larger than the within-group variance — a meaningful signal above random noise.
- The p-value of 0.02 means there is only a **2% probability** of observing this F-statistic if all groups were truly equal.

**Important Note:**
ANOVA only tells us **that** a difference exists — it does NOT tell us **which groups differ**. To find that, we need a **post-hoc test** (e.g., Tukey HSD, Bonferroni).

## Q7. Handling Missing Data in Repeated Measures ANOVA

**Methods to Handle Missing Data:**

1. **Listwise Deletion (Complete Case Analysis)**:
   - Remove any subject with missing data entirely.
   - Simple but wastes data and can bias results if data is not MCAR (Missing Completely at Random).
   - Consequence: Reduced sample size, loss of statistical power.

2. **Mean Imputation**:
   - Replace missing values with the group/overall mean.
   - Consequence: Underestimates variance, distorts correlations between timepoints.

3. **Last Observation Carried Forward (LOCF)**:
   - Use the subject's last observed value for missing timepoints.
   - Used in clinical trials but assumes no change — unrealistic.

4. **Multiple Imputation (MI)**:
   - Creates multiple plausible complete datasets, analyzes each, and pools results.
   - Best method when data is MAR (Missing at Random) — preserves variance and relationships.

5. **Mixed-Effects Models (LMM)**:
   - Uses all available data without requiring complete cases.
   - Gold standard for repeated measures with missing data.

**Key Consequence:**
The chosen method can dramatically change conclusions — using listwise deletion on data that is not MCAR introduces bias and inflates Type I or Type II errors.

## Q8. Common Post-Hoc Tests after ANOVA

**Why Post-Hoc Tests?**
ANOVA only tells us at least one group is different. Post-hoc tests identify **which specific groups differ**.

| Post-Hoc Test | When to Use |
|---|---|
| **Tukey HSD** | Equal group sizes, comparing all pairs — most common |
| **Bonferroni** | Few planned comparisons, very conservative |
| **Scheffe** | Unequal sample sizes, complex comparisons |
| **Dunnett** | Comparing all groups to a single control group |
| **Games-Howell** | Unequal variances or unequal sample sizes |
| **LSD (Fisher)** | Liberal test — use only when ANOVA is significant |

**Example Scenario:**
A pharmaceutical company tests three drug doses (Low, Medium, High) on pain relief. One-way ANOVA shows significant difference (F=8.2, p=0.001). We need Tukey HSD to find out:
- Is Low vs Medium significant?
- Is Low vs High significant?
- Is Medium vs High significant?

## Q9. One-Way ANOVA — Weight loss comparison across 3 diets

In [3]:
import numpy as np
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import pandas as pd

np.random.seed(42)

# Simulated weight loss data for 3 diets (50 participants → ~17 each)
diet_A = np.random.normal(loc=5.0, scale=1.5, size=17)  # avg 5 lbs
diet_B = np.random.normal(loc=7.0, scale=1.5, size=17)  # avg 7 lbs
diet_C = np.random.normal(loc=4.0, scale=1.5, size=16)  # avg 4 lbs

f_stat, p_value = stats.f_oneway(diet_A, diet_B, diet_C)
alpha = 0.05

print("===== One-Way ANOVA — Diet Weight Loss =====")
print(f"Diet A: mean={diet_A.mean():.2f}, n={len(diet_A)}")
print(f"Diet B: mean={diet_B.mean():.2f}, n={len(diet_B)}")
print(f"Diet C: mean={diet_C.mean():.2f}, n={len(diet_C)}")
print(f"\nF-statistic : {f_stat:.4f}")
print(f"p-value     : {p_value:.6f}")
print(f"Alpha       : {alpha}")

if p_value < alpha:
    print("\nResult: REJECT H₀ — Significant difference in mean weight loss between diets.")
    print("\n--- Tukey HSD Post-Hoc Test ---")
    all_data   = np.concatenate([diet_A, diet_B, diet_C])
    all_labels = ['Diet A']*len(diet_A) + ['Diet B']*len(diet_B) + ['Diet C']*len(diet_C)
    tukey = pairwise_tukeyhsd(endog=all_data, groups=all_labels, alpha=0.05)
    print(tukey)
else:
    print("\nResult: FAIL TO REJECT H₀ — No significant difference between diets.")

===== One-Way ANOVA — Diet Weight Loss =====
Diet A: mean=4.87, n=17
Diet B: mean=6.64, n=17
Diet C: mean=3.46, n=16

F-statistic : 20.9322
p-value     : 0.000000
Alpha       : 0.05

Result: REJECT H₀ — Significant difference in mean weight loss between diets.

--- Tukey HSD Post-Hoc Test ---


Multiple Comparison of Means - Tukey HSD, FWER=0.05 
group1 group2 meandiff p-adj   lower   upper  reject
----------------------------------------------------
Diet A Diet B   1.7684  0.002  0.5904  2.9463   True
Diet A Diet C  -1.4193 0.0165 -2.6155  -0.223   True
Diet B Diet C  -3.1876    0.0 -4.3839 -1.9914   True
----------------------------------------------------


## Q10. Two-Way ANOVA — Software Program x Experience Level on task completion time

In [4]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.formula.api import ols

np.random.seed(42)

# 30 employees: 3 programs x 2 experience levels x 5 each
n = 5
data = pd.DataFrame({
    'Time': np.concatenate([
        np.random.normal(30, 5, n),  # Prog A, Novice
        np.random.normal(20, 5, n),  # Prog A, Experienced
        np.random.normal(25, 5, n),  # Prog B, Novice
        np.random.normal(18, 5, n),  # Prog B, Experienced
        np.random.normal(35, 5, n),  # Prog C, Novice
        np.random.normal(28, 5, n),  # Prog C, Experienced
    ]),
    'Program':    ['A']*n*2 + ['B']*n*2 + ['C']*n*2,
    'Experience': (['Novice']*n + ['Experienced']*n) * 3
})

model = ols('Time ~ C(Program) + C(Experience) + C(Program):C(Experience)', data=data).fit()
anova_table = sm.stats.anova_lm(model, typ=2)

print("===== Two-Way ANOVA — Software x Experience =====")
print(anova_table)
print("\nInterpretation:")
alpha = 0.05
for idx in anova_table.index[:-1]:
    p = anova_table.loc[idx, 'PR(>F)']
    sig = 'SIGNIFICANT' if p < alpha else 'NOT significant'
    print(f"  {idx:<35} → p={p:.4f} → {sig}")

===== Two-Way ANOVA — Software x Experience =====
                              sum_sq    df          F        PR(>F)
C(Program)                896.205136   2.0  27.461999  6.252100e-07
C(Experience)             490.464810   1.0  30.058173  1.231560e-05
C(Program):C(Experience)   18.700715   2.0   0.573037  5.713372e-01
Residual                  391.612478  24.0        NaN           NaN

Interpretation:
  C(Program)                          → p=0.0000 → SIGNIFICANT
  C(Experience)                       → p=0.0000 → SIGNIFICANT
  C(Program):C(Experience)            → p=0.5713 → NOT significant


## Q11. Two-sample t-test — New vs Traditional teaching method + Post-hoc

In [5]:
import numpy as np
from scipy import stats

np.random.seed(42)

# 100 students: 50 control, 50 experimental
control      = np.random.normal(loc=70, scale=10, size=50)  # traditional
experimental = np.random.normal(loc=76, scale=10, size=50)  # new method

# H0: μ_control = μ_experimental
# H1: μ_control ≠ μ_experimental
t_stat, p_value = stats.ttest_ind(control, experimental)
alpha = 0.05

print("===== Two-Sample t-Test — Teaching Methods =====")
print(f"Control Group      → mean={control.mean():.2f}, std={control.std():.2f}, n={len(control)}")
print(f"Experimental Group → mean={experimental.mean():.2f}, std={experimental.std():.2f}, n={len(experimental)}")
print(f"\nt-statistic : {t_stat:.4f}")
print(f"p-value     : {p_value:.4f}")
print(f"Alpha       : {alpha}")

if p_value < alpha:
    print("\nResult: REJECT H₀ — Significant difference between the two teaching methods.")
    print(f"\nPost-Hoc Interpretation (only 2 groups):")
    print(f"  The experimental group (mean={experimental.mean():.2f}) scored significantly")
    print(f"  higher than the control group (mean={control.mean():.2f}).")
    print("  → New teaching method is significantly more effective.")
else:
    print("\nResult: FAIL TO REJECT H₀ — No significant difference between teaching methods.")

===== Two-Sample t-Test — Teaching Methods =====
Control Group      → mean=67.75, std=9.24, n=50
Experimental Group → mean=76.18, std=8.66, n=50

t-statistic : -4.6615
p-value     : 0.0000
Alpha       : 0.05

Result: REJECT H₀ — Significant difference between the two teaching methods.

Post-Hoc Interpretation (only 2 groups):
  The experimental group (mean=76.18) scored significantly
  higher than the control group (mean=67.75).
  → New teaching method is significantly more effective.


## Q12. Repeated Measures ANOVA — Daily sales across 3 stores + Post-hoc

In [6]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

np.random.seed(42)

# 30 days, 3 stores — same days observed (repeated measures)
n_days  = 30
store_A = np.random.normal(loc=500, scale=50, size=n_days)
store_B = np.random.normal(loc=550, scale=50, size=n_days)
store_C = np.random.normal(loc=480, scale=50, size=n_days)

# Using one-way ANOVA as approximation (pingouin for full repeated measures)
f_stat, p_value = stats.f_oneway(store_A, store_B, store_C)
alpha = 0.05

print("===== Repeated Measures ANOVA — Store Daily Sales =====")
print(f"Store A: mean=${store_A.mean():.2f}")
print(f"Store B: mean=${store_B.mean():.2f}")
print(f"Store C: mean=${store_C.mean():.2f}")
print(f"\nF-statistic : {f_stat:.4f}")
print(f"p-value     : {p_value:.4f}")
print(f"Alpha       : {alpha}")

if p_value < alpha:
    print("\nResult: REJECT H₀ — Significant difference in daily sales between stores.")
    print("\n--- Tukey HSD Post-Hoc Test ---")
    all_sales  = np.concatenate([store_A, store_B, store_C])
    all_labels = ['Store A']*n_days + ['Store B']*n_days + ['Store C']*n_days
    tukey = pairwise_tukeyhsd(endog=all_sales, groups=all_labels, alpha=0.05)
    print(tukey)
else:
    print("\nResult: FAIL TO REJECT H₀ — No significant difference in sales between stores.")

===== Repeated Measures ANOVA — Store Daily Sales =====
Store A: mean=$490.59
Store B: mean=$543.94
Store C: mean=$480.64

F-statistic : 15.6747
p-value     : 0.0000
Alpha       : 0.05

Result: REJECT H₀ — Significant difference in daily sales between stores.

--- Tukey HSD Post-Hoc Test ---


  Multiple Comparison of Means - Tukey HSD, FWER=0.05   
 group1  group2 meandiff p-adj   lower    upper   reject
--------------------------------------------------------
Store A Store B  53.3492 0.0001  24.3572  82.3413   True
Store A Store C  -9.9484 0.6928 -38.9405  19.0437  False
Store B Store C -63.2976    0.0 -92.2897 -34.3056   True
--------------------------------------------------------
